In [36]:
import pandas as pd
import requests as req
from bs4 import BeautifulSoup as B, NavigableString, Tag, Comment, Doctype
import time, re

In [37]:
# links = []
# titles = []
# for i in range(1, 172):
#     url = f'https://www.ettoday.net/news_search/doSearch.php?keywords=%E8%98%87%E8%8A%B1%E5%85%AC%E8%B7%AF&idx=1&page={i}'
#     print(f'抓取第 {i} 頁：{url}')

#     resp = req.get(url)
#     if resp.status_code != 200 :
#         print('Error status_code')
#         continue

#     soup = B(resp.text, 'html.parser')
#     for div_ in soup.find_all('div', class_='archive clearfix'):
#         h2_tag = div_.find('h2')
#         if h2_tag:
#             a_tag = h2_tag.find('a', href=True)
#             if a_tag:
#                 link = a_tag['href']
#                 title = a_tag.get_text(strip=True)
#                 links.append(link)
#                 titles.append(title)
#     time.sleep(1)

# data = pd.DataFrame({'Link': links, 'Title': titles})
# data.to_csv('ettoday_蘇花公路.csv', index=False)

In [38]:
data = pd.read_csv('ettoday_蘇花公路.csv')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1702 entries, 0 to 1701
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Link    1702 non-null   object
 1   Title   1702 non-null   object
dtypes: object(2)
memory usage: 26.7+ KB


In [39]:
data.shape

(1702, 2)

In [40]:
data['Context'] = ''; data['PublishTime'] = ''

failed_urls = []

for i, url in enumerate(data['Link']):
    print(f'[{i+1}/{len(data)}] 處理中：{url}')

    try:
        resp = req.get(url)

        if resp.status_code != 200:
            print(f'無法存取：{resp.status_code}')
            data.at[i, 'Context'] = ''
            data.at[i, 'PublishTime'] = ''
            failed_urls.append(url)
            continue

        soup = B(resp.text, 'html.parser')

        time_tag = soup.find('time', class_='date')
        publish_time = time_tag['datetime'][:19] if time_tag and 'datetime' in time_tag.attrs else ''

        data.at[i, 'PublishTime'] = publish_time

        article_span = soup.find_all('p')
        context = []
        EXCLUDE_P_CLASSES = {'et_165dashboard__info', 'note', 'summary', 'figcaption', 'figcaption'}
        EXCLUDE_P_PARENTS = {'et_social_2', 'et_social_3', 'et_epaper_box_pc_sidebar'}
        EXCLUDE_TEXT_KEYWORDS = ['圖／公路局提供', '圖／', '記者拍攝', '照片來源', '記者', '首次上稿', '更新時間', '圖文／鏡週刊', '▲', '公路局提醒用路人']
        for p in article_span:
            if 'class' in p.attrs and any(cls in EXCLUDE_P_CLASSES for cls in p['class']):
                continue
            if any(p.find_parent(class_=parent_class) for parent_class in EXCLUDE_P_PARENTS):
                continue
            if any(keyword in p.get_text() for keyword in ['更多鏡週刊報導']):
                continue
            if p.find(['strong', 'span'], class_='figcaption'):
                continue
            text = p.get_text(strip=True)
            if any(keyword in text for keyword in EXCLUDE_TEXT_KEYWORDS):
                continue

            if text:
                context.append(text)
        data.at[i, 'Context'] = '\n'.join(context)
        
    except Exception as e:
        print(f'發生錯誤:{e}')
        data.at[i, 'Context'] = ''
        data.at[i, 'PublishTime'] = ''
        failed_urls.append(url)

    data.to_csv('ettoday_蘇花公路_temp.csv', index=False)
    time.sleep(1) 

data.to_csv('ettoday_蘇花公路_full.csv', index=False)
with open('ettoday_蘇花公路_failed.txt', 'w', encoding='utf-8') as f:
    for url in failed_urls:
        f.write(url+'\n')
print('全部完成, 已儲存完整資料和失敗網址')

[1/1702] 處理中：https://www.ettoday.net/news/20250622/2982821.htm
[2/1702] 處理中：https://www.ettoday.net/news/20250616/2979408.htm
[3/1702] 處理中：https://www.ettoday.net/news/20250609/2975007.htm
[4/1702] 處理中：https://www.ettoday.net/news/20250609/2975010.htm
[5/1702] 處理中：https://www.ettoday.net/news/20250605/2973031.htm
[6/1702] 處理中：https://www.ettoday.net/news/20250520/2964098.htm
[7/1702] 處理中：https://www.ettoday.net/news/20250520/2964063.htm
[8/1702] 處理中：https://www.ettoday.net/news/20250520/2964060.htm
[9/1702] 處理中：https://www.ettoday.net/news/20250520/2964044.htm
[10/1702] 處理中：https://www.ettoday.net/news/20250520/2963953.htm
[11/1702] 處理中：https://www.ettoday.net/news/20250520/2963573.htm
[12/1702] 處理中：https://www.ettoday.net/news/20250518/2962837.htm
[13/1702] 處理中：https://www.ettoday.net/news/20250518/2962748.htm
[14/1702] 處理中：https://www.ettoday.net/news/20250513/2959428.htm
[15/1702] 處理中：https://www.ettoday.net/news/20250507/2956180.htm
[16/1702] 處理中：https://www.ettoday.net/news/202504

In [41]:
# links = []
# titles = []
# for i in range(1, 107):
#     url = f'https://www.ettoday.net/news_search/doSearch.php?keywords=%E8%98%87%E8%8A%B1%E6%94%B9&idx=1&page={i}'
#     print(f'抓取第 {i} 頁：{url}')

#     resp = req.get(url)
#     if resp.status_code != 200 :
#         print('Error status_code')
#         continue

#     soup = B(resp.text, 'html.parser')
#     for div_ in soup.find_all('div', class_='archive clearfix'):
#         h2_tag = div_.find('h2')
#         if h2_tag:
#             a_tag = h2_tag.find('a', href=True)
#             if a_tag:
#                 link = a_tag['href']
#                 title = a_tag.get_text(strip=True)
#                 links.append(link)
#                 titles.append(title)
#     time.sleep(1)

# data = pd.DataFrame({'Link': links, 'Title': titles})
# data.to_csv('ettoday_蘇花改.csv', index=False)

In [42]:
data = pd.read_csv('ettoday_蘇花改.csv')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1060 entries, 0 to 1059
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Link    1060 non-null   object
 1   Title   1060 non-null   object
dtypes: object(2)
memory usage: 16.7+ KB


In [43]:
data.shape

(1060, 2)

In [44]:
data['Context'] = ''; data['PublishTime'] = ''

failed_urls = []

for i, url in enumerate(data['Link']):
    print(f'[{i+1}/{len(data)}] 處理中：{url}')

    try:
        resp = req.get(url)

        if resp.status_code != 200:
            print(f'無法存取：{resp.status_code}')
            data.at[i, 'Context'] = ''
            data.at[i, 'PublishTime'] = ''
            failed_urls.append(url)
            continue

        soup = B(resp.text, 'html.parser')

        time_tag = soup.find('time', class_='date')
        publish_time = time_tag['datetime'][:19] if time_tag and 'datetime' in time_tag.attrs else ''
        data.at[i, 'PublishTime'] = publish_time

        article_span = soup.find_all('p')
        context = []
        EXCLUDE_P_CLASSES = {'et_165dashboard__info', 'note', 'summary', 'figcaption', 'figcaption'}
        EXCLUDE_P_PARENTS = {'et_social_2', 'et_social_3', 'et_epaper_box_pc_sidebar'}
        EXCLUDE_TEXT_KEYWORDS = ['圖／公路局提供', '圖／', '記者拍攝', '照片來源', '記者', '首次上稿', '更新時間', '圖文／鏡週刊', '▲', '公路局提醒用路人']
        for p in article_span:
            if 'class' in p.attrs and any(cls in EXCLUDE_P_CLASSES for cls in p['class']):
                continue
            if any(p.find_parent(class_=parent_class) for parent_class in EXCLUDE_P_PARENTS):
                continue
            if any(keyword in p.get_text() for keyword in ['更多鏡週刊報導']):
                continue
            if p.find(['strong', 'span'], class_='figcaption'):
                continue
            text = p.get_text(strip=True)
            if any(keyword in text for keyword in EXCLUDE_TEXT_KEYWORDS):
                continue

            if text:
                context.append(text)
        data.at[i, 'Context'] = '\n'.join(context)
        
    except Exception as e:
        print(f'發生錯誤:{e}')
        data.at[i, 'Context'] = ''
        data.at[i, 'PublishTime'] = ''
        failed_urls.append(url)

    data.to_csv('ettoday_蘇花改_temp.csv', index=False)
    time.sleep(1) 

data.to_csv('ettoday_蘇花改_full.csv', index=False)
with open('ettoday_蘇花改_failed.txt', 'w', encoding='utf-8') as f:
    for url in failed_urls:
        f.write(url+'\n')
print('全部完成, 已儲存完整資料和失敗網址')

[1/1060] 處理中：https://www.ettoday.net/news/20250529/2969155.htm
[2/1060] 處理中：https://www.ettoday.net/news/20250520/2963573.htm
[3/1060] 處理中：https://www.ettoday.net/news/20250518/2962837.htm
[4/1060] 處理中：https://www.ettoday.net/news/20250513/2959428.htm
[5/1060] 處理中：https://www.ettoday.net/news/20250512/2959139.htm
[6/1060] 處理中：https://www.ettoday.net/news/20250507/2956101.htm
[7/1060] 處理中：https://www.ettoday.net/news/20250425/2949734.htm
[8/1060] 處理中：https://www.ettoday.net/news/20250418/2945609.htm
[9/1060] 處理中：https://www.ettoday.net/news/20250402/2936375.htm
[10/1060] 處理中：https://www.ettoday.net/news/20250401/2935725.htm
[11/1060] 處理中：https://www.ettoday.net/news/20250331/2934750.htm
[12/1060] 處理中：https://www.ettoday.net/news/20250327/2932621.htm
[13/1060] 處理中：https://www.ettoday.net/news/20250310/2922848.htm
[14/1060] 處理中：https://www.ettoday.net/news/20250228/2916985.htm
[15/1060] 處理中：https://www.ettoday.net/news/20250227/2916641.htm
[16/1060] 處理中：https://www.ettoday.net/news/202502